<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Debug the experiment

**[Debug the experiment](https://learning.nextia-ai.com/courses/deep-learning/m04/debug-the-experiment/)** · Deep Learning and Transformers Explained · Module 4, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find the cause of a failed training run in a fixed order (labels, inputs, a tiny batch, gradients), and repair a seeded training failure with evidence from the curves and the gradients.

| | |
|---|---|
| **Time** | About 45 minutes, with the module practice. Every run takes seconds. |
| **Needs** | An internet connection for the setup cells. No account. PyTorch (preinstalled in Colab and Kaggle). |
| **You should know** | Backpropagation, from [Backpropagation and autograd](https://learning.nextia-ai.com/courses/deep-learning/m03/backpropagation-and-autograd/). The curve shapes and overfitting, from [Learning rate and optimizers](https://learning.nextia-ai.com/courses/deep-learning/m04/learning-rate-and-optimizers/) and [Overfitting controls](https://learning.nextia-ai.com/courses/deep-learning/m04/overfitting-controls/). Cross-validation, from the machine learning course. |
| **You do not need** | A graphics card or a local project. The setup below downloads the tickets and the course's code. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m04/debug-the-experiment/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M04-L03-debug-the-experiment/debug-the-experiment-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: the list of the planted wrong labels

This lesson checks how well each method finds the wrong labels in the training data. The file `data/split_manifest.json` lists them. Run the next cell: it downloads the file (a few kilobytes) and checks its checksum. You should see `Ready: split_manifest.json`.

In [ ]:
# The IDs of the 70 planted wrong labels, in data/split_manifest.json.
path = Path("data/split_manifest.json")
if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5":
    download(LABS + "data/split_manifest.json", path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5":
        raise SystemExit("split_manifest.json has the wrong checksum. Delete it and run this cell again.")
print("Ready: split_manifest.json")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The problem: a symptom with many causes

First, the bag-of-words tensors, as in Module 3. Run the cell.

In [ ]:
import torch
from torch import nn
from ticketnet import *       # the course's code: Vocab, bag_of_words, BagOfWordsNet, train() ...

train_rows, valid_rows = load_split("data/train.csv"), load_split("data/valid.csv")
vocab = Vocab([r["text"] for r in train_rows])          # words from the training tickets only
x_train, x_valid = bag_of_words(train_rows, vocab), bag_of_words(valid_rows, vocab)
y_train, y_valid = labels(train_rows), labels(valid_rows)
print("x_train", tuple(x_train.shape), " x_valid", tuple(x_valid.shape))

> **Check.** You should see `x_train (4696, 2864)  x_valid (600, 2864)`.

This test gives each training ticket the team of another, random ticket.

> **Predict.** What happens to the training accuracy and the validation accuracy over 10 epochs? Then run the cell.

In [ ]:
g = torch.Generator().manual_seed(0)
shuffled_labels = y_train[torch.randperm(len(y_train), generator=g)]   # each ticket gets a random ticket's team
set_seed(0)
model = BagOfWordsNet(len(vocab))
history = train(model, x_train, shuffled_labels, x_valid, y_valid, epochs=10, log=None)
for row in history[::3]:
    print(f"epoch {row['epoch']:>2}: train accuracy {row['train_acc']:.1%}, valid accuracy {row['valid_acc']:.1%}")

> **Check.** You should see:
>
> `epoch  1: train accuracy 28.4%, valid accuracy 25.5%`  
> `epoch  4: train accuracy 52.0%, valid accuracy 24.8%`  
> `epoch  7: train accuracy 68.7%, valid accuracy 20.7%`  
> `epoch 10: train accuracy 80.0%, valid accuracy 19.2%`  

Training goes up, validation stays at chance (20% for five teams): the network memorises labels with no meaning. Remember this picture.

The checklist, in order: **1** look at examples and labels, **2** check the inputs, **3** overfit a tiny batch, **4** check the gradients. Tune only after all four pass. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m04/debug-the-experiment/#the-checklist) has the diagram.

## 2. Check 1: look at examples and labels

> **Your turn.** Print the first five training tickets with their team: `for r in train_rows[:5]: print(r["team"], "|", r["text"][:70])`. Read them: does each team fit its ticket? There is no check cell.

In [ ]:
# print the first five training tickets with their team
...

1.5% of the training labels are wrong on purpose: 70 tickets, listed in `data/split_manifest.json` so that you can check each method. Tomás's idea: train on all rows and look at the training tickets that the network gets wrong. Run the cell.

In [ ]:
import json
noisy = set(json.load(open("data/split_manifest.json"))["noisy_train_ids"])   # the 70 planted wrong labels
set_seed(0)
full = BagOfWordsNet(len(vocab))
train(full, x_train, y_train, x_valid, y_valid, epochs=20, log=None)   # trained on all training rows
full.eval()
with torch.no_grad():
    wrong = (full(x_train).argmax(1) != y_train).nonzero().flatten().tolist()
print("training tickets the network gets wrong:", len(wrong))
print("of these, planted wrong labels:", sum(train_rows[i]["ticket_id"] in noisy for i in wrong))

> **Check.** You should see:
>
> `training tickets the network gets wrong: 2`  
> `of these, planted wrong labels: 2`  

The network memorised the other 68 wrong labels, so it cannot point to them. Judge a label only with a model that did not learn it: **out-of-fold** predictions, as in cross-validation. Run the cell. It trains 5 networks and takes about 10 seconds.

In [ ]:
order = torch.randperm(len(train_rows), generator=torch.Generator().manual_seed(0))
probs = torch.zeros(len(train_rows), len(TEAMS))
for k in range(5):                                   # 5 folds
    held_out = order[k::5]                            # every 5th ticket: this fold
    kept = torch.tensor(sorted(set(order.tolist()) - set(held_out.tolist())))
    set_seed(k)
    model = BagOfWordsNet(len(vocab))
    train(model, x_train[kept], y_train[kept], x_train[held_out], y_train[held_out], epochs=8, log=None)
    model.eval()
    with torch.no_grad():
        probs[held_out] = model(x_train[held_out]).softmax(1)   # scored by a model that never saw them
p_given = probs[torch.arange(len(train_rows)), y_train]       # the probability of the label in the file
suspects = p_given.argsort()[:50].tolist()                    # the 50 least likely labels
for i in suspects[:3]:
    r = train_rows[i]
    print(r["ticket_id"], "label:", r["team"], " network:", TEAMS[int(probs[i].argmax())],
          f" p(label) {float(p_given[i]):.3f} ", r["text"][:60] + "...")

> **Check.** You should see:
>
> `T-60623 label: warranty  network: delivery  p(label) 0.000  I had a very helpful chat with my neighbour last week, and s...`  
> `T-60552 label: payment  network: returns  p(label) 0.000  I got the wrong item for my log store order LK-160935. The b...`  
> `T-62277 label: warranty  network: delivery  p(label) 0.000  Hi there! So I was really looking forward to getting my hamm...`  

> **Your turn.** How many of the 70 planted wrong labels are among the 100 lowest values of `p_given`? Use `p_given.argsort()[:100]`, the ticket IDs in `train_rows`, and the set `noisy`. Store the count in `found_in_100`, then run the check cell.

In [ ]:
found_in_100 = ...  # planted wrong labels among the 100 least likely labels
print(found_in_100)

In [ ]:
expect('found_in_100', found_in_100, 69)

Now the 50 suspects. Run the cell.

In [ ]:
print("planted wrong labels:", len(noisy))
print("among the 50 suspects:", sum(train_rows[i]["ticket_id"] in noisy for i in suspects))
print("among the 100 lowest:", sum(train_rows[i]["ticket_id"] in noisy for i in p_given.argsort()[:100].tolist()))

> **Check.** You should see:
>
> `planted wrong labels: 70`  
> `among the 50 suspects: 49`  
> `among the 100 lowest: 69`  

49 of the 50 suspects are planted wrong labels: from 2 to 49 with the same network, used in a different way. A person (Grace) should read the suspects and correct the labels; do not delete them automatically.

## 3. Check 2: the inputs

Print the shape and the range of the values. Run the cell.

In [ ]:
print("input shape:", tuple(x_train.shape), " values from", int(x_train.min()), "to", int(x_train.max()))
print("after log1p, the network sees values up to", round(float(torch.log1p(x_train).max()), 2))

> **Check.** You should see:
>
> `input shape: (4696, 2864)  values from 0 to 16`  
> `after log1p, the network sees values up to 2.83`  

Then look for **leakage**. Tomás builds the vocabulary from the training and the validation tickets. Run the cell.

In [ ]:
def unknown_share(rows, vocab):
    ids = [t for r in rows for t in vocab.encode(r["text"])]
    return sum(t == 1 for t in ids) / len(ids)            # ID 1 is <unk>

leaky = Vocab([r["text"] for r in train_rows + valid_rows])  # the mistake: valid words in the vocabulary
print("train only:   ", len(vocab), "words,", f"{unknown_share(valid_rows, vocab):.1%} of valid tokens unknown")
print("train + valid:", len(leaky), "words,", f"{unknown_share(valid_rows, leaky):.1%} of valid tokens unknown")

> **Check.** You should see:
>
> `train only:    2864 words, 1.2% of valid tokens unknown`  
> `train + valid: 2987 words, 0.8% of valid tokens unknown`  

> **Your turn.** How many words did the validation tickets add to the vocabulary? Store the number in `added_words`, then run the check cell.

In [ ]:
added_words = ...  # the words in leaky that are not in vocab
print(added_words)

In [ ]:
expect('added_words', added_words, 123)

No curve shows this leak. You find it only by checking how each input was built: every step that learns from data uses the training tickets only.

## 4. Check 3: overfit a tiny batch

A working network and training loop must learn 20 tickets by heart. Run the cell.

In [ ]:
set_seed(0)
tiny_model = BagOfWordsNet(len(vocab))
tiny = torch.arange(20)                               # the first 20 training tickets
tiny_history = train(tiny_model, x_train[tiny], y_train[tiny], x_train[tiny], y_train[tiny],
                     epochs=60, batch_size=20, lr=0.01, log=None)
for row in tiny_history[::10] + [tiny_history[-1]]:
    print(f"step {row['epoch']:>2}: loss {row['train_loss']:.4f}, accuracy {row['train_acc']:.0%}")

> **Check.** You should see:
>
> `step  1: loss 1.3483, accuracy 80%`  
> `step 11: loss 0.0256, accuracy 100%`  
> `step 21: loss 0.0007, accuracy 100%`  
> `step 31: loss 0.0001, accuracy 100%`  
> `step 41: loss 0.0000, accuracy 100%`  
> `step 51: loss 0.0000, accuracy 100%`  
> `step 60: loss 0.0000, accuracy 100%`  

> **Your turn.** The output shows every tenth step. Find the **first** step where the accuracy is 100%, from `tiny_history`, and store it in `first_perfect_step`. Then run the check cell.

In [ ]:
first_perfect_step = ...  # the first row of tiny_history with train_acc == 1.0
print(first_perfect_step)

In [ ]:
expect('first_perfect_step', first_perfect_step, 5)

If this check fails, the bug is in the code: the model, the loss, the optimizer, or the pairing of tickets and labels.

## 5. Check 4: the gradients

**Exploding gradients.** Adam with learning rate 10, for four steps. Run the cell.

In [ ]:
set_seed(0)
model = BagOfWordsNet(len(vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=10)     # 10 000 times the default
loss_fn = nn.CrossEntropyLoss()
for step in range(4):
    batch = torch.arange(step * 64, step * 64 + 64)
    optimizer.zero_grad()
    loss = loss_fn(model(x_train[batch]), y_train[batch])
    loss.backward()
    print(f"step {step}: loss {loss.item():10.3f}   gradient norm, first layer {model.layers[0].weight.grad.norm().item():8.3f}")
    optimizer.step()

> **Check.** You should see:
>
> `step 0: loss      1.616   gradient norm, first layer    0.354`  
> `step 1: loss  33688.914   gradient norm, first layer  209.409`  
> `step 2: loss  24560.604   gradient norm, first layer  121.913`  
> `step 3: loss  19077.770   gradient norm, first layer  101.325`  

**Vanishing gradients.** A stack of 10 linear layers with sigmoid, then with ReLU. Run the cell.

In [ ]:
def gradient_norms(activation):
    """A stack of 10 linear layers; one backward pass; the gradient size of each layer."""
    set_seed(0)
    layers = [nn.Linear(len(vocab), 64), activation()]
    for _ in range(8):
        layers += [nn.Linear(64, 64), activation()]
    stack = nn.Sequential(*layers, nn.Linear(64, 5))
    loss = nn.CrossEntropyLoss()(stack(torch.log1p(x_train[:256])), y_train[:256])
    loss.backward()
    return [layer.weight.grad.norm().item() for layer in stack if isinstance(layer, nn.Linear)]

for name, activation in [("sigmoid", nn.Sigmoid), ("ReLU", nn.ReLU)]:
    norms = gradient_norms(activation)
    print(f"{name:<8} first layer {norms[0]:.2e}   last layer {norms[-1]:.2e}")

> **Check.** You should see:
>
> `sigmoid  first layer 1.45e-08   last layer 6.24e-01`  
> `ReLU     first layer 1.31e-04   last layer 2.05e-02`  

Sigmoid's slope is never more than 0.25, so the blame shrinks at every layer on its way back.

> **Your turn.** Call `gradient_norms(nn.ReLU)` and find the layer (1 to 10) with the **smallest** gradient norm. Store its number in `smallest_layer`, then run the check cell.

In [ ]:
relu_norms = gradient_norms(nn.ReLU)
smallest_layer = ...  # 1 to 10: the layer with the smallest norm
print(smallest_layer)

In [ ]:
expect('smallest_layer', smallest_layer, 2)

With ReLU the gradients shrink too, but far less than with sigmoid. Module 6 adds residual connections and normalization, two more defences.

## 6. Module practice: repair a seeded training failure

Tomás wrote `broken_train.py`. It runs without an error, but the router is bad. It has **two bugs**.

The next cell writes the script into your folder. To repair a bug, change the script **in this cell** and run it again. Run it now. You should see `Writing broken_train.py` (or `Overwriting`).

In [ ]:
%%writefile broken_train.py
"""Tomás's training script for the ticket router. It runs, but the router is bad.

Run it from the folder ticket-router (it needs data/ and ticketnet.py):
    python broken_train.py
"""
import torch
from torch import nn

from ticketnet import BagOfWordsNet, Vocab, bag_of_words, evaluate, labels, load_split, set_seed

torch.set_num_threads(2)
LEARNING_RATE = 10
EPOCHS = 8

set_seed(0)
train_rows = load_split("data/train.csv")
valid_rows = load_split("data/valid.csv")
vocab = Vocab([r["text"] for r in train_rows])
x_train, y_train = bag_of_words(train_rows, vocab), labels(train_rows)
x_valid, y_valid = bag_of_words(valid_rows, vocab), labels(valid_rows)

# Mix the tickets once, so that the teams are not in file order.
mix = torch.randperm(len(x_train))
x_train = x_train[mix]

model = BagOfWordsNet(len(vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(1, EPOCHS + 1):
    model.train()
    order = torch.randperm(len(x_train))
    for i in range(0, len(order), 64):
        batch = order[i:i + 64]
        optimizer.zero_grad()
        loss = loss_fn(model(x_train[batch]), y_train[batch])
        loss.backward()
        optimizer.step()
    grad = model.layers[0].weight.grad.norm().item()
    train_loss, train_acc = evaluate(model, x_train, y_train)
    valid_loss, valid_acc = evaluate(model, x_valid, y_valid)
    print(f"epoch {epoch}: train loss {train_loss:8.3f}  acc {train_acc:6.1%} | "
          f"valid loss {valid_loss:8.3f}  acc {valid_acc:6.1%} | gradient norm {grad:8.3f}")

The next cell runs the script. `%run` also keeps its variables, so you can inspect `model`, `x_train`, `y_train` or `mix` afterwards. Run it.

In [ ]:
%run broken_train.py

> **Check.** Before any repair, you should see:
>
> `epoch 1: train loss   77.472  acc  21.4% | valid loss   21.518  acc  21.3% | gradient norm    8.843`  
> `epoch 2: train loss   40.332  acc  17.2% | valid loss   13.111  acc  17.0% | gradient norm    9.303`  
> `epoch 3: train loss    2.642  acc  27.4% | valid loss    2.148  acc  27.5% | gradient norm    0.000`  
> `epoch 4: train loss    1.866  acc  27.4% | valid loss    2.065  acc  27.5% | gradient norm    0.000`  
> `epoch 5: train loss    1.735  acc  14.5% | valid loss    1.938  acc  14.7% | gradient norm    0.000`  
> `epoch 6: train loss    1.605  acc  27.4% | valid loss    1.811  acc  27.5% | gradient norm    0.000`  
> `epoch 7: train loss    1.726  acc  21.4% | valid loss    1.933  acc  21.0% | gradient norm    0.000`  
> `epoch 8: train loss    1.893  acc  20.1% | valid loss    2.094  acc  20.0% | gradient norm    0.000`  

Work through the checklist:

1. Read the output with the curve shapes and the gradient sizes of this module. Which check does it point to?
2. Repair one bug in the `%%writefile` cell, run that cell, then run `%run broken_train.py` again.
3. Compare the new output with the old one. Repeat until the router reaches about 98% on validation.

> **Hint.** If the training accuracy rises while the validation accuracy stays near 20%, go back to check 1: print a few tickets with their labels after the mix, for example `print(TEAMS[int(y_train[0])], train_rows[int(mix[0])]["text"][:70])`.

The check cell reads `valid_acc`, the validation accuracy of the last epoch of your last run.

In [ ]:
expect("validation accuracy after 8 epochs", round(valid_acc, 3), 0.98, tolerance=0.01)

> **Check.** After both repairs, you should see:
>
> `epoch 1: train loss    0.536  acc  96.4% | valid loss    0.559  acc  97.7% | gradient norm    0.601`  
> `epoch 2: train loss    0.209  acc  97.4% | valid loss    0.212  acc  98.2% | gradient norm    0.812`  
> `epoch 3: train loss    0.136  acc  98.0% | valid loss    0.139  acc  98.5% | gradient norm    0.202`  
> `epoch 4: train loss    0.100  acc  98.4% | valid loss    0.110  acc  98.5% | gradient norm    0.170`  
> `epoch 5: train loss    0.078  acc  98.7% | valid loss    0.095  acc  98.2% | gradient norm    0.108`  
> `epoch 6: train loss    0.061  acc  98.8% | valid loss    0.082  acc  98.3% | gradient norm    0.391`  
> `epoch 7: train loss    0.048  acc  99.2% | valid loss    0.076  acc  98.8% | gradient norm    0.437`  
> `epoch 8: train loss    0.040  acc  99.3% | valid loss    0.083  acc  98.0% | gradient norm    0.085`  

> **Your turn.** Write your diagnosis record in the cell below: double-click it and fill in the table. For each bug, give the evidence as numbers from the output, before and after the repair.

The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m04/debug-the-experiment/#module-practice) has the model answer.

| | Bug 1 | Bug 2 |
|---|---|---|
| Symptom | | |
| Evidence (numbers from the output) | | |
| Cause (the line in the script) | | |
| Repair | | |
| Evidence after the repair | | |

## Reset

To start the practice again, run the `%%writefile` cell of the original notebook again (or open the notebook again from the lesson page). If a data file or `ticketnet.py` changed by mistake, delete the folder `ticket-router` and run the setup cells again.

## Recap

- When a run fails silently, check in order: examples and labels, inputs, a tiny batch, gradients. Tune only after that.
- Judge a label only with a model that did not learn it: out-of-fold predictions found 49 of the 70 wrong labels among 50 suspects.
- Training up and validation at chance means meaningless labels. A vocabulary built with validation tickets is leakage.
- A huge loss or a zero gradient norm points to the learning rate; tiny gradients in early layers point to the activation.

The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m04/debug-the-experiment/) has the checklist diagram, the module check and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Tokens and embeddings](https://learning.nextia-ai.com/courses/deep-learning/m05/tokens-and-embeddings/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m04/debug-the-experiment/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.